In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import timm
import os
import sys
import gc
import matplotlib.pyplot as plt

# Set absolute paths
PROJECT_ROOT = r'd:\Projects\ML_Algorithms\batchsize'
IMAGE_PATH = r'E:\covidx'
TRAIN_CSV = os.path.join(IMAGE_PATH, 'covidx_merged.csv')

# Add project root and functions to path
if PROJECT_ROOT not in sys.path: sys.path.append(PROJECT_ROOT)
functions_path = os.path.join(PROJECT_ROOT, "functions")
if functions_path not in sys.path: sys.path.append(functions_path)

from dataset import COVIDCXNetDataset
from train import train
from evaluation import plot_results


from batch_size_experiment import run_batch_size_experiments
from visualize_results import visualize_batch_size_results
from find_max_batch import find_max_batch_size
from sample_experiment import run_sample_experiment

c:\Users\Furkan\Miniconda3\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
batch_sizes = [8, 16, 32, 64, 128, 211, 256, 512, 1024, 2048, 4096, 8192]
num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

ROOT_DATA_DIR = os.path.dirname(IMAGE_PATH) # E:\\
train_ds = COVIDCXNetDataset(TRAIN_CSV, ROOT_DATA_DIR, transform=transform, split='train')
val_ds = COVIDCXNetDataset(TRAIN_CSV, ROOT_DATA_DIR, transform=transform, split='val')


Using device: cuda
Original samples: 67863
Valid AP/PA samples: 53691
Loaded 53691 samples for split 'train'
Class distribution: PA=20388, AP=33303
Original samples: 8473
Valid AP/PA samples: 4186
Loaded 4186 samples for split 'val'
Class distribution: PA=1275, AP=2911


In [3]:
find_max_batch_size(device = device,model_name='resnet50')


Searching for maximum Batch Size in range [8, 4096] for model 'resnet50'...

Testing Batch Size: 2052...
OOM ❌

Testing Batch Size: 1029...
OOM ❌

Testing Batch Size: 518...
OOM ❌

Testing Batch Size: 262...
SUCCESS | Allocated: 563.96 MB | Reserved: 15542.00 MB | Peak: 21796.94 MB

Testing Batch Size: 390...
OOM ❌

Testing Batch Size: 326...
OOM ❌

Testing Batch Size: 294...
OOM ❌

Testing Batch Size: 278...
OOM ❌

Testing Batch Size: 270...
SUCCESS | Allocated: 566.47 MB | Reserved: 20790.00 MB | Peak: 22464.14 MB

Testing Batch Size: 274...
OOM ❌

Testing Batch Size: 272...
OOM ❌

Testing Batch Size: 271...
OOM ❌

>>> Maximum working Batch Size found: 270 <<<
>>> Recommended Safe Batch Size: 216 <<<


216

In [5]:
run_sample_experiment(image_path=IMAGE_PATH,batch_sizes=batch_sizes,sample_train_size='full',sample_val_size='full',model_name='resnet50',device=device)

Device: cuda
Setting up dataset...
Original samples: 67863
Valid AP/PA samples: 53691
Loaded 53691 samples for split 'train'
Class distribution: PA=20388, AP=33303
Sample Train Size: 53691
Sample Val Size: 0

--- Testing Batch Size: 8 ---


Success! Max Memory: 1005.83 MB

--- Testing Batch Size: 16 ---


Success! Max Memory: 2026.67 MB

--- Testing Batch Size: 32 ---


Success! Max Memory: 3333.77 MB

--- Testing Batch Size: 64 ---


Success! Max Memory: 5991.22 MB

--- Testing Batch Size: 128 ---


...

OOM Error for Batch Size 1024

--- Testing Batch Size: 2048 ---


OOM Error for Batch Size 2048

--- Testing Batch Size: 4096 ---


OOM Error for Batch Size 4096

--- Testing Batch Size: 8192 ---


OOM Error for Batch Size 8192

--- Final Results ---
Batch Size 8: Success (Mem: 1005.83 MB)
Batch Size 16: Success (Mem: 2026.67 MB)
Batch Size 32: Success (Mem: 3333.77 MB)
Batch Size 64: Success (Mem: 5991.22 MB)
Batch Size 128: Success (Mem: 11292.47 MB)
Batch Size 211: Success (Mem: 18154.05 MB)
Batch Size 256: Success (Mem: 21880.43 MB)
Batch Size 512: OOM
Batch Size 1024: OOM
Batch Size 2048: OOM
Batch Size 4096: OOM
Batch Size 8192: OOM


{8: 'Success (Mem: 1005.83 MB)',
 16: 'Success (Mem: 2026.67 MB)',
 32: 'Success (Mem: 3333.77 MB)',
 64: 'Success (Mem: 5991.22 MB)',
 128: 'Success (Mem: 11292.47 MB)',
 211: 'Success (Mem: 18154.05 MB)',
 256: 'Success (Mem: 21880.43 MB)',
 512: 'OOM',
 1024: 'OOM',
 2048: 'OOM',
 4096: 'OOM',
 8192: 'OOM'}

In [ ]:
results = run_batch_size_experiments(batch_sizes=batch_sizes, num_epochs=5, project_root=PROJECT_ROOT, model_name='resnet50', train_ds=train_ds, val_ds=val_ds)

In [ ]:
visualize_batch_size_results(PROJECT_ROOT,model='resnet50')